<a href="https://colab.research.google.com/github/Akobabs/IRIS-FINGERPRINT-PSO-SVM/blob/main/script1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

=======================================================================
MULTIMODAL BIOMETRIC RECOGNITION SYSTEM
Iris + Fingerprint | Feature-Level Fusion | PCA + PSO-SVM
=======================================================================
"""
DATASETS USED:
  PRIMARY (Homologous):
    Data/IRIS and FINGERPRINT DATASET/
      └── {1..45}/
            ├── Fingerprint/   → {id}__M/F_{Left/Right}_{finger}_finger.BMP  (10 files)
            ├── left/          → aeval{1..5}.bmp   (5 left iris images)
            └── right/         → aevar{1..5}.bmp   (5 right iris images)

  SECONDARY (Chimeric / Cross-validation):
    Data/.../Iris_Recognition/MMU-Iris-Database/
      └── {subject}_{eye}_{img}.png   (eye: 1=left, 2=right; img: 1-5)
    Data/.../Finger_print_recognition/L3SF_V2/L3-SF/R1..R5/
      └── {subject}_{finger}_{img}.png  (finger: 1-2; img: 1-5; across R1-R5 sub-folders)

HOW TO USE:
  1. Upload the entire 'Data' folder to your Google Drive
  2. Copy-paste each cell into a new Google Colab notebook
  3. Update BASE_DIR in Cell 2 if your folder path is different
  4. Run cells sequentially
=======================================================================
"""

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



# =====================================================================
# CELL 1: Install Dependencies & Imports
# =====================================================================

In [2]:
!pip install pyswarms scikit-learn opencv-python-headless tqdm matplotlib numpy

import os
import re
import glob
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
import pyswarms as ps

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted. TF version:", tf.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.1 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. TF version: 2.20.0


# =====================================================================
# CELL 2: Configuration — UPDATE BASE_DIR TO MATCH YOUR DRIVE LOCATION
# =====================================================================
# Assumes you placed the project folder at the root of your Drive:
## My Drive / IRIS-FINGERPRINT-PSO-SVM / Data / ...

In [3]:
BASE_DIR = "/content/drive/MyDrive/IRIS-FINGERPRINT-PSO-SVM/Data"

# --- Primary Homologous Dataset ---
HOMOLOGOUS_DIR = os.path.join(BASE_DIR, "IRIS and FINGERPRINT DATASET")
NUM_SUBJECTS    = 45   # Subjects 1 – 45

# --- Secondary Chimeric Dataset ---
MMU_IRIS_DIR = os.path.join(
    BASE_DIR,
    "Face_Fingerprint_iris_recognition_dataset",
    "Iris_Recognition",
    "MMU-Iris-Database"
)
L3SF_FP_DIR = os.path.join(
    BASE_DIR,
    "Face_Fingerprint_iris_recognition_dataset",
    "Finger_print_recognition",
    "L3SF_V2", "L3-SF"
)
L3SF_SUBFOLDERS = ["R1", "R2", "R3", "R4", "R5"]  # 5 repetitions

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Configuration loaded.")
print(f"  Homologous dir : {HOMOLOGOUS_DIR}")
print(f"  MMU Iris dir   : {MMU_IRIS_DIR}")
print(f"  L3-SF FP dir   : {L3SF_FP_DIR}")

Configuration loaded.
  Homologous dir : /content/drive/MyDrive/IRIS-FINGERPRINT-PSO-SVM/Data/IRIS and FINGERPRINT DATASET
  MMU Iris dir   : /content/drive/MyDrive/IRIS-FINGERPRINT-PSO-SVM/Data/Face_Fingerprint_iris_recognition_dataset/Iris_Recognition/MMU-Iris-Database
  L3-SF FP dir   : /content/drive/MyDrive/IRIS-FINGERPRINT-PSO-SVM/Data/Face_Fingerprint_iris_recognition_dataset/Finger_print_recognition/L3SF_V2/L3-SF


# =====================================================================
# CELL 3: Feature Extractor — MobileNetV2 (pretrained, no top)
# =====================================================================

In [4]:
print("Loading MobileNetV2 feature extractor …")
_feat_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
_feat_model.trainable = False
print(f"Feature vector size: {_feat_model.output_shape[-1]}")   # 1280-D


def extract_features(bgr_img: np.ndarray) -> np.ndarray:
    """
    Takes a BGR image (OpenCV), resizes to 224x224, and extracts a
    1280-D feature vector using the frozen MobileNetV2 backbone.
    """
    if bgr_img is None:
        return None
    rgb  = cv2.cvtColor(cv2.resize(bgr_img, (224, 224)), cv2.COLOR_BGR2RGB)
    arr  = np.expand_dims(rgb.astype(np.float32), axis=0)
    arr  = preprocess_input(arr)          # MobileNetV2 preprocessing
    feat = _feat_model.predict(arr, verbose=0)
    return feat.flatten()                  # shape: (1280,)

Loading MobileNetV2 feature extractor …


/tmp/ipykernel_9927/1600012840.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  _feat_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Feature vector size: 1280


# =====================================================================
# CELL 4: Image Preprocessing Helpers
# =====================================================================

In [5]:
def preprocess_iris_img(path: str) -> np.ndarray:
    """
    Read a BMP/PNG iris image and apply CLAHE for contrast enhancement.
    Returns a 3-channel BGR image ready for the CNN extractor.
    """
    img = cv2.imread(path)
    if img is None:
        print(f"  [WARN] Cannot read iris image: {path}")
        return None
    gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


def preprocess_fp_img(path: str) -> np.ndarray:
    """
    Read a BMP/PNG fingerprint image and enhance ridges with
    adaptive thresholding.
    Returns a 3-channel BGR image ready for the CNN extractor.
    """
    img = cv2.imread(path)
    if img is None:
        print(f"  [WARN] Cannot read fingerprint image: {path}")
        return None
    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)

# =====================================================================
# CELL 5: Data Loader — Primary Homologous Dataset
# =====================================================================

In [6]:
"""
Directory structure (confirmed):
  IRIS and FINGERPRINT DATASET/
    {subject_id}/           ← e.g. "1", "2", … "45"
      Fingerprint/
        {id}__M_Left_index_finger.BMP   (10 .BMP files total)
        {id}__M_Left_little_finger.BMP
        … (all 10 fingers)
      left/
        aeval1.bmp  aeval2.bmp  aeval3.bmp  aeval4.bmp  aeval5.bmp
      right/
        aevar1.bmp  aevar2.bmp  aevar3.bmp  aevar4.bmp  aevar5.bmp

Strategy:
  Per subject we have 10 iris images (5 left + 5 right) and 10 fingerprint
  images (one per finger).  We form 10 fused feature samples per subject
  by pairing iris[i] ↔ fingerprint[i] (sorted order, mod 10).
"""

def load_homologous_dataset(data_dir: str, num_subjects: int = 45):
    X_list, y_list = [], []

    print(f"\n{'='*60}")
    print("Loading PRIMARY HOMOLOGOUS dataset …")
    print(f"{'='*60}")

    for subj_id in tqdm(range(1, num_subjects + 1), desc="Subjects"):
        subj_path = os.path.join(data_dir, str(subj_id))
        if not os.path.isdir(subj_path):
            print(f"  [WARN] Subject folder missing: {subj_path}")
            continue

        # --- Collect fingerprint paths ---
        fp_dir    = os.path.join(subj_path, "Fingerprint")
        fp_paths  = sorted([p for p in glob.glob(os.path.join(fp_dir, "*.BMP"))
                             if not os.path.basename(p).lower().startswith("desktop")])

        # --- Collect iris paths ---
        left_paths  = sorted(glob.glob(os.path.join(subj_path, "left",  "aeval*.bmp")))
        right_paths = sorted(glob.glob(os.path.join(subj_path, "right", "aevar*.bmp")))
        iris_paths  = left_paths + right_paths     # 10 total (5+5)

        if len(fp_paths) == 0 or len(iris_paths) == 0:
            print(f"  [WARN] Subject {subj_id}: missing images "
                  f"(FP={len(fp_paths)}, Iris={len(iris_paths)})")
            continue

        num_pairs = min(len(fp_paths), len(iris_paths))   # should be 10

        for i in range(num_pairs):
            iris_img = preprocess_iris_img(iris_paths[i])
            fp_img   = preprocess_fp_img(fp_paths[i % len(fp_paths)])

            iris_feat = extract_features(iris_img)
            fp_feat   = extract_features(fp_img)

            if iris_feat is None or fp_feat is None:
                continue

            # Feature-level fusion: simple concatenation → 2560-D vector
            fused = np.concatenate([iris_feat, fp_feat])
            X_list.append(fused)
            y_list.append(subj_id)

    X = np.array(X_list)
    y = np.array(y_list)
    print(f"\nHomologous dataset loaded: {X.shape[0]} samples, "
          f"{len(np.unique(y))} subjects, feature dim={X.shape[1]}")
    return X, y


# =====================================================================
# CELL 6: Data Loader — Secondary Chimeric Dataset (MMU Iris + L3-SF FP)
# =====================================================================

In [7]:

"""
MMU-Iris-Database (flat folder):
  Filename: {subject}_{eye}_{img}.png
    subject : 1–148  (we use first 45)
    eye     : 1 = left, 2 = right
    img     : 1–5
  So subject 1, left eye = 1_1_1.png, 1_1_2.png, … 1_1_5.png
     subject 1, right eye= 1_2_1.png, 1_2_2.png, … 1_2_5.png

L3-SF Fingerprint (split across R1/R2/R3/R4/R5):
  Filename: {subject}_{finger}_{img}.png
    subject : 1–148  (we use first 45)
    finger  : 1 or 2
    img     : 1–5
  Each sub-folder is an independent acquisition repetition.
  We pool all images for subject across all sub-folders.

Chimeric pairing: subject i in MMU ↔ subject i in L3-SF.
"""

def get_mmu_iris_paths(iris_dir: str, subject_id: int):
    """
    Returns sorted list of iris PNG paths for a given subject (1-indexed).
    Collects both eyes.
    """
    pattern = os.path.join(iris_dir, f"{subject_id}_*.png")
    paths = sorted(glob.glob(pattern))
    return paths   # typically 10 files (2 eyes × 5 images)


def get_l3sf_fp_paths(fp_base_dir: str, subfolders: list, subject_id: int):
    """
    Returns sorted list of fingerprint PNG paths for a given subject,
    pooling across all R1–R5 sub-folders.
    """
    paths = []
    for sf in subfolders:
        pattern = os.path.join(fp_base_dir, sf, f"{subject_id}_*.png")
        paths.extend(glob.glob(pattern))
    return sorted(paths)


def load_chimeric_dataset(iris_dir: str, fp_base_dir: str,
                           subfolders: list, num_subjects: int = 45):
    X_list, y_list = [], []

    print(f"\n{'='*60}")
    print("Loading SECONDARY CHIMERIC dataset …")
    print(f"{'='*60}")

    for subj_id in tqdm(range(1, num_subjects + 1), desc="Subjects"):
        iris_paths = get_mmu_iris_paths(iris_dir, subj_id)
        fp_paths   = get_l3sf_fp_paths(fp_base_dir, subfolders, subj_id)

        if len(iris_paths) == 0 or len(fp_paths) == 0:
            print(f"  [WARN] Subject {subj_id}: no images found "
                  f"(iris={len(iris_paths)}, fp={len(fp_paths)})")
            continue

        # For chimeric, take up to 10 pairs
        num_pairs = min(len(iris_paths), len(fp_paths), 10)

        for i in range(num_pairs):
            iris_img = preprocess_iris_img(iris_paths[i])
            fp_img   = preprocess_fp_img(fp_paths[i % len(fp_paths)])

            iris_feat = extract_features(iris_img)
            fp_feat   = extract_features(fp_img)

            if iris_feat is None or fp_feat is None:
                continue

            fused = np.concatenate([iris_feat, fp_feat])
            X_list.append(fused)
            # Offset labels so they don't overlap with homologous (1-45)
            y_list.append(subj_id + 100)

    X = np.array(X_list)
    y = np.array(y_list)
    print(f"\nChimeric dataset loaded: {X.shape[0]} samples, "
          f"{len(np.unique(y))} subjects, feature dim={X.shape[1]}")
    return X, y

# =====================================================================
# CELL 7: Load Both Datasets
# =====================================================================

In [8]:
X_homo, y_homo = load_homologous_dataset(HOMOLOGOUS_DIR, NUM_SUBJECTS)
X_chim, y_chim = load_chimeric_dataset(MMU_IRIS_DIR, L3SF_FP_DIR,
                                        L3SF_SUBFOLDERS, NUM_SUBJECTS)

print(f"\nSummary:")
print(f"  Homologous : X={X_homo.shape}, y={y_homo.shape}")
print(f"  Chimeric   : X={X_chim.shape}, y={y_chim.shape}")


Loading PRIMARY HOMOLOGOUS dataset …


Subjects:   4%|▍         | 2/45 [00:50<15:06, 21.08s/it]

  [WARN] Subject 2: missing images (FP=10, Iris=0)


Subjects:   7%|▋         | 3/45 [00:51<08:12, 11.73s/it]

  [WARN] Subject 3: missing images (FP=10, Iris=0)


...

Subjects:  96%|█████████▌| 43/45 [00:15<00:00,  6.06it/s]

  [WARN] Subject 42: no images found (iris=0, fp=50)
  [WARN] Subject 43: no images found (iris=0, fp=50)


Subjects: 100%|██████████| 45/45 [00:15<00:00,  2.86it/s]

  [WARN] Subject 44: no images found (iris=0, fp=50)
  [WARN] Subject 45: no images found (iris=0, fp=50)


IndexError: tuple index out of range

# =====================================================================
# CELL 8: Visualise Sample Images (sanity check)
# =====================================================================

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle("Sample Images from Both Datasets", fontsize=14)

# Homologous — Subject 1
subj1_left_iris = sorted(glob.glob(
    os.path.join(HOMOLOGOUS_DIR, "1", "left", "aeval*.bmp")))[0]
subj1_fp = sorted(glob.glob(
    os.path.join(HOMOLOGOUS_DIR, "1", "Fingerprint", "*.BMP")))[0]

axes[0, 0].imshow(cv2.cvtColor(cv2.imread(subj1_left_iris), cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("Homo – S1 Left Iris (raw)")
axes[0, 1].imshow(cv2.cvtColor(preprocess_iris_img(subj1_left_iris), cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("Homo – S1 Left Iris (CLAHE)")
axes[0, 2].imshow(cv2.cvtColor(cv2.imread(subj1_fp), cv2.COLOR_BGR2RGB))
axes[0, 2].set_title("Homo – S1 Fingerprint (raw)")
axes[0, 3].imshow(cv2.cvtColor(preprocess_fp_img(subj1_fp), cv2.COLOR_BGR2RGB))
axes[0, 3].set_title("Homo – S1 Fingerprint (enhanced)")

# Chimeric — Subject 1
chim_iris = get_mmu_iris_paths(MMU_IRIS_DIR, 1)[0]
chim_fp   = get_l3sf_fp_paths(L3SF_FP_DIR, L3SF_SUBFOLDERS, 1)[0]

axes[1, 0].imshow(cv2.cvtColor(cv2.imread(chim_iris), cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("Chim – MMU Iris (raw)")
axes[1, 1].imshow(cv2.cvtColor(preprocess_iris_img(chim_iris), cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("Chim – MMU Iris (CLAHE)")
axes[1, 2].imshow(cv2.cvtColor(cv2.imread(chim_fp), cv2.COLOR_BGR2RGB))
axes[1, 2].set_title("Chim – L3-SF FP (raw)")
axes[1, 3].imshow(cv2.cvtColor(preprocess_fp_img(chim_fp), cv2.COLOR_BGR2RGB))
axes[1, 3].set_title("Chim – L3-SF FP (enhanced)")

for ax in axes.flat: ax.axis('off')
plt.tight_layout()
plt.show()

# =====================================================================
# CELL 9: PCA Dimensionality Reduction (on Homologous data)
# =====================================================================

In [ ]:
print("\nNormalising features (StandardScaler) …")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_homo)

print("Applying PCA (retaining 98% variance) …")
pca = PCA(n_components=0.98, svd_solver='full', random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)

print(f"  Original dim  : {X_scaled.shape[1]}")
print(f"  Reduced dim   : {X_pca.shape[1]}")
print(f"  Variance kept : {pca.explained_variance_ratio_.sum() * 100:.2f}%")

# Plot cumulative explained variance
plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_) * 100, color='steelblue')
plt.axhline(98, color='red', linestyle='--', label='98% threshold')
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance (%)")
plt.title("PCA Explained Variance")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# =====================================================================
# CELL 10: Train / Test Split
# =====================================================================

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y_homo,
    test_size=0.30,
    stratify=y_homo,
    random_state=RANDOM_SEED
)
print(f"Train samples: {X_train.shape[0]}")
print(f"Test  samples: {X_test.shape[0]}")

# =====================================================================
# CELL 11: PSO-SVM — Fitness Function & Optimiser
# =====================================================================

In [ ]:

def svm_fitness(params: np.ndarray,
                X_tr: np.ndarray,
                y_tr: np.ndarray,
                cv: int = 3) -> np.ndarray:
    """
    PSO fitness function.
    params shape: (n_particles, 2)  → column 0 = C, column 1 = gamma
    Returns error rate (1 - accuracy) for each particle.
    """
    n_particles = params.shape[0]
    scores = np.zeros(n_particles)
    for i in range(n_particles):
        C_val     = float(params[i, 0])
        gamma_val = float(params[i, 1])
        model = SVC(kernel='rbf', C=C_val, gamma=gamma_val, random_state=RANDOM_SEED)
        cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='accuracy', n_jobs=-1)
        scores[i] = 1.0 - cv_scores.mean()   # PSO minimises → return error
    return scores


def optimise_svm_pso(X_tr, y_tr,
                      n_particles=15, n_iter=30, cv=3):
    """
    Run GlobalBest PSO to find optimal (C, gamma) for SVM-RBF.
    Search space: C ∈ [0.1, 1000],  gamma ∈ [1e-4, 1.0]
    """
    # Bounds: [[C_min, gamma_min], [C_max, gamma_max]]
    bounds = (np.array([0.1,  1e-4]),
              np.array([1000.0, 1.0]))

    options = {
        'c1': 0.5,   # cognitive weight (personal best)
        'c2': 0.3,   # social weight (global best)
        'w':  0.9    # inertia weight
    }

    optimizer = ps.single.GlobalBestPSO(
        n_particles=n_particles,
        dimensions=2,
        options=options,
        bounds=bounds
    )

    print(f"\nRunning PSO ({n_particles} particles × {n_iter} iterations) …")
    cost, best_pos = optimizer.optimize(
        lambda p: svm_fitness(p, X_tr, y_tr, cv=cv),
        iters=n_iter,
        verbose=True
    )

    best_C     = best_pos[0]
    best_gamma = best_pos[1]
    print(f"\n  Best error rate  : {cost:.4f}  →  CV accuracy ≈ {(1-cost)*100:.2f}%")
    print(f"  Optimal C        : {best_C:.4f}")
    print(f"  Optimal gamma    : {best_gamma:.6f}")
    return best_C, best_gamma, optimizer


best_C, best_gamma, pso_optimizer = optimise_svm_pso(
    X_train, y_train,
    n_particles=15,
    n_iter=30,
    cv=3
)

# Plot PSO convergence
plt.figure(figsize=(8, 4))
plt.plot(pso_optimizer.cost_history, color='darkorange')
plt.xlabel("Iteration"); plt.ylabel("Best Cost (Error Rate)")
plt.title("PSO Convergence Curve")
plt.grid(True); plt.tight_layout(); plt.show()

# =====================================================================
# CELL 12: Train Final SVM with Optimised Parameters
# =====================================================================


In [ ]:
print("\nTraining final SVM-RBF with PSO-optimised hyperparameters …")
final_svm = SVC(
    kernel='rbf',
    C=best_C,
    gamma=best_gamma,
    probability=True,
    random_state=RANDOM_SEED
)
final_svm.fit(X_train, y_train)
print("  Training complete.")

# =====================================================================
# CELL 13: Evaluation — Accuracy, FAR, FRR, Confusion Matrix
# =====================================================================

In [ ]:
y_pred = final_svm.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred, labels=np.unique(y_homo))
TP = np.diag(cm).astype(float)
FP = (cm.sum(axis=0) - TP).astype(float)
FN = (cm.sum(axis=1) - TP).astype(float)
TN = (cm.sum() - (TP + FP + FN)).astype(float)

# Avoid division by zero
FAR_per_class = np.where((FP + TN) > 0, FP / (FP + TN), 0.0)
FRR_per_class = np.where((FN + TP) > 0, FN / (FN + TP), 0.0)
EER_approx    = (FAR_per_class.mean() + FRR_per_class.mean()) / 2.0

print("\n" + "="*55)
print(f"  FINAL TEST ACCURACY  : {acc * 100:.2f}%")
print(f"  Avg FAR (per class)  : {FAR_per_class.mean() * 100:.4f}%")
print(f"  Avg FRR (per class)  : {FRR_per_class.mean() * 100:.4f}%")
print(f"  Approx EER           : {EER_approx * 100:.4f}%")
print("="*55)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix heatmap (works well for ≤ 45 classes)
fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=np.unique(y_homo))
disp.plot(ax=ax, xticks_rotation=90, colorbar=True, cmap='Blues')
ax.set_title("Confusion Matrix — Homologous Dataset")
plt.tight_layout(); plt.show()



# =====================================================================
# CELL 14: Cross-Dataset Validation with Chimeric Data
# =====================================================================

In [ ]:
"""
Apply the same scaler+PCA fitted on homologous data to the
chimeric data and evaluate.  This tests generalisation across sensors.
"""
print("\n" + "="*55)
print("Cross-dataset validation on CHIMERIC data …")
print("="*55)

X_chim_scaled = scaler.transform(X_chim)
X_chim_pca    = pca.transform(X_chim_scaled)

# Re-map chimeric labels (101-145) to 1-45 for fair comparison
y_chim_mapped = y_chim - 100

# Predict on chimeric data using the trained SVM
y_chim_pred = final_svm.predict(X_chim_pca)
y_chim_pred_mapped = y_chim_pred   # labels are already 1-45 in SVM space

# Accuracy: how many chimeric subjects are correctly identified?
chim_acc = accuracy_score(y_chim_mapped, y_chim_pred)
print(f"\n  Chimeric Test Accuracy: {chim_acc * 100:.2f}%")
print("  (Note: lower accuracy expected — different sensors/populations)")


# =====================================================================
# CELL 15: Save Results Summary
# =====================================================================

In [ ]:
import json, datetime

results = {
    "timestamp"         : datetime.datetime.now().isoformat(),
    "pso_best_C"        : round(best_C, 4),
    "pso_best_gamma"    : round(best_gamma, 6),
    "homologous_acc_pct": round(acc * 100, 2),
    "avg_FAR_pct"       : round(FAR_per_class.mean() * 100, 4),
    "avg_FRR_pct"       : round(FRR_per_class.mean() * 100, 4),
    "approx_EER_pct"    : round(EER_approx * 100, 4),
    "chimeric_acc_pct"  : round(chim_acc * 100, 2),
    "pca_components"    : int(X_pca.shape[1]),
    "train_samples"     : int(X_train.shape[0]),
    "test_samples"      : int(X_test.shape[0]),
}

save_path = "/content/drive/MyDrive/IRIS-FINGERPRINT-PSO-SVM/results_summary.json"
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {save_path}")
print(json.dumps(results, indent=2))
